# Notebook 21 – Mini Assessment Answers



## 1. Deep learning fundamentals
Artificial intelligence is the broad field of intelligent computer systems. Machine learning is a way to build AI systems that learn from data. Deep learning is machine learning based on neural networks with multiple layers. Traditional ML often needs manually designed features; deep learning can learn features from raw data. Common applications include image recognition, speech, translation, and recommendations. Supervised learning uses labeled examples; unsupervised learning finds patterns in unlabeled data.

## 2. Neurons, weights, bias, and perceptrons
A neuron calculates a weighted sum plus a bias, then applies an activation: $z=um_i w_i x_i+b$. Inputs carry information, weights control input influence, and bias shifts the threshold. A perceptron uses a step activation and can represent linearly separable rules such as AND and OR. A single perceptron cannot solve XOR because XOR classes cannot be separated by one straight line.

In [3]:
import numpy as np

def perceptron(inputs, weights, bias):
    score = np.dot(inputs, weights) + bias
    return int(score >= 0)

and_weights = [1, 1]
or_weights = [1, 1]
for first, second in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    and_result = perceptron([first, second], and_weights, -1.5)
    or_result = perceptron([first, second], or_weights, -0.5)
    print((first, second), 'AND:', and_result, 'OR:', or_result)

(0, 0) AND: 0 OR: 0
(0, 1) AND: 0 OR: 1
(1, 0) AND: 0 OR: 1
(1, 1) AND: 1 OR: 1


## 3. Activation functions
Activations add nonlinearity; without them, any stack of linear layers is still one linear transformation. Sigmoid maps to (0, 1), tanh to (-1, 1), ReLU to [0, infinity), and softmax turns class scores into probabilities that sum to 1. Sigmoid and tanh can have vanishing gradients at extreme inputs. ReLU is efficient but can produce dead neurons; Leaky ReLU keeps a small negative-side slope.

In [4]:
import torch
scores = torch.tensor([-2.0, 0.0, 2.0])
print('Sigmoid:', torch.sigmoid(scores))
print('Tanh:', torch.tanh(scores))
print('ReLU:', torch.relu(scores))
print('Leaky ReLU:', torch.nn.functional.leaky_relu(scores, negative_slope=0.01))
print('Softmax:', torch.softmax(torch.tensor([1.0, 2.0, 0.0]), dim=0))

Sigmoid: tensor([0.1192, 0.5000, 0.8808])
Tanh: tensor([-0.9640,  0.0000,  0.9640])
ReLU: tensor([0., 0., 2.])
Leaky ReLU: tensor([-0.0200,  0.0000,  2.0000])
Softmax: tensor([0.2447, 0.6652, 0.0900])


## 4. Forward propagation
Forward propagation moves input data through linear layers and activations to produce an output. For input 2, hidden calculation ReLU(3 × 2 + 1) gives 7; an output calculation 2 × 7 − 1 gives 13. Hidden layers learn intermediate features, and the output layer produces scores or predictions.

In [5]:
hidden_value = torch.relu(torch.tensor(3.0 * 2 + 1))
prediction = 2 * hidden_value - 1
print('Hidden activation:', hidden_value.item(), 'Prediction:', prediction.item())

Hidden activation: 7.0 Prediction: 13.0


## 5. Loss functions
A loss measures prediction error. MAE averages absolute errors and is less sensitive to outliers; MSE squares errors and penalizes large misses more; RMSE is the square root of MSE and returns to the target's units. Binary cross-entropy is used for binary probabilities, BCE-with-logits combines sigmoid and BCE stably, and multiclass cross-entropy expects raw class logits and integer class labels.

In [6]:
targets = torch.tensor([1.0, 0.0])
predictions = torch.tensor([0.8, 0.4])
print('MAE:', torch.mean(torch.abs(targets - predictions)).item())
print('MSE:', torch.mean((targets - predictions) ** 2).item())
print('RMSE:', torch.sqrt(torch.mean((targets - predictions) ** 2)).item())
print('BCE:', torch.nn.functional.binary_cross_entropy(predictions, targets).item())
print('BCE with logits:', torch.nn.BCEWithLogitsLoss()(torch.tensor([1.4, -0.4]), targets).item())
class_logits = torch.tensor([[1.0, 2.0, 0.0]])
class_target = torch.tensor([1])
print('Multiclass cross-entropy:', torch.nn.CrossEntropyLoss()(class_logits, class_target).item())

MAE: 0.30000001192092896
MSE: 0.10000000149011612
RMSE: 0.3162277638912201
BCE: 0.3669845759868622
BCE with logits: 0.3667163550853729
Multiclass cross-entropy: 0.40760594606399536


## 6. Gradients and backpropagation
A gradient indicates how changing a parameter changes the loss. Backpropagation applies the chain rule from the loss toward earlier layers. For $y=x^2$ at $x=3$, the derivative is $2x=6$. PyTorch autograd records operations, `.backward()` calculates gradients, and leaf tensor gradients are available in `.grad`.

In [7]:
value = torch.tensor(3.0, requires_grad=True)
output = value ** 2
output.backward()
print('Gradient:', value.grad.item())
with torch.no_grad():
    inference_output = value ** 2
print('Tracks gradient in no_grad:', inference_output.requires_grad)

Gradient: 6.0
Tracks gradient in no_grad: False


## 7. Gradient descent and optimizers
Gradient descent updates parameters as $w_{new}=w_{old}-ta
abla L$, where $ta$ is the learning rate. If $w=2$, gradient is 3, and learning rate is 0.1, the updated weight is 1.7. Batch descent uses all examples per update, stochastic descent uses one, and mini-batch descent uses a small group. SGD is simple, momentum smooths updates using past gradients, RMSprop adapts step sizes, and Adam combines momentum with adaptive scaling. A learning rate that is too high may diverge; one that is too low learns slowly.

In [8]:
weight = 2.0
gradient = 3.0
learning_rate = 0.1
weight -= learning_rate * gradient
print('Updated weight:', weight)
model = torch.nn.Linear(2, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print('Optimizer:', type(optimizer).__name__)

Updated weight: 1.7
Optimizer: Adam


## 8. PyTorch tensors and devices
A tensor is a multidimensional numeric array with shape, dimensions, and dtype. PyTorch tensors can run on CPU or a supported GPU and can track gradients, unlike ordinary NumPy arrays. Use indexing and slicing to select values, reshape/view to change dimensions without changing values, and matrix multiplication to compute layer transformations. `torch.no_grad()` is useful for inference because it avoids storing a gradient graph.

In [9]:
import numpy as np
numpy_values = np.array([[1, 2], [3, 4]], dtype=np.float32)
tensor_values = torch.from_numpy(numpy_values)
print('Shape:', tensor_values.shape, 'dimensions:', tensor_values.ndim, 'dtype:', tensor_values.dtype)
print('First row:', tensor_values[0], 'sum:', tensor_values.sum().item())
print('Reshape:', tensor_values.reshape(4))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tensor_values = tensor_values.to(device)
print('Device:', tensor_values.device)
print('Back to NumPy:', tensor_values.cpu().numpy())

Shape: torch.Size([2, 2]) dimensions: 2 dtype: torch.float32
First row: tensor([1., 2.]) sum: 10.0
Reshape: tensor([1., 2., 3., 4.])
Device: cpu
Back to NumPy: [[1. 2.]
 [3. 4.]]


## 9. Dataset and DataLoader
A Dataset provides one example and its label through `__getitem__()` and reports its size through `__len__()`. A DataLoader batches samples and can shuffle them, making training more memory-efficient and reducing dependence on sample order.

In [10]:
from torch.utils.data import TensorDataset, DataLoader
features = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
labels = torch.tensor([0, 1, 0])
dataset = TensorDataset(features, labels)
batch_features, batch_labels = next(iter(DataLoader(dataset, batch_size=2, shuffle=True)))
print('Batch shapes:', batch_features.shape, batch_labels.shape)

Batch shapes: torch.Size([2, 2]) torch.Size([2])


## 10. ANN architecture and training
An ANN for tabular data uses fully connected layers. `nn.Linear(in_features, out_features)` applies a learned weight matrix and bias vector. Training repeats: get a batch, clear old gradients, calculate predictions and loss, call backward, and update parameters. An epoch is one pass through the training set; validation loss measures performance on held-out data.

In [11]:
import torch.nn as nn
ann = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
print(ann)
print('Parameters:', sum(parameter.numel() for parameter in ann.parameters()))
model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
inputs = torch.tensor([[1.0, 2.0]])
target = torch.tensor([[1.0]])
optimizer.zero_grad()
loss = (model(inputs) - target).square().mean()
loss.backward()
optimizer.step()
print('One training-step loss:', loss.item())

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=2, bias=True)
)
Parameters: 58
One training-step loss: 2.4295897483825684


## 11. CNNs and image dimensions
A CNN is suited to images because convolution filters learn local spatial patterns and share weights across locations. A convolution creates feature maps; stride controls movement, padding controls border handling, pooling downsamples, and flattening converts feature maps for a fully connected classifier. For a 32×32 image, a 3×3 convolution with stride 1 and no padding gives 30×30; 2×2 max pooling with stride 2 gives 15×15.

In [12]:
cnn = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2), nn.Flatten(),
    nn.Linear(8 * 4 * 4, 10)
)
image_batch = torch.randn(5, 1, 8, 8)
print('CNN output shape:', cnn(image_batch).shape)

CNN output shape: torch.Size([5, 10])


## 12. Overfitting and regularization
Overfitting means training performance is much better than validation performance. Underfitting means both are poor. Dropout randomly disables activations during training; weight decay penalizes large weights; batch normalization stabilizes intermediate activations; early stopping restores the best validation checkpoint. Compare training and validation curves to determine whether a method actually improved generalization.

In [13]:
regularized_model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Dropout(0.2), nn.Linear(8, 1))
optimizer = torch.optim.Adam(regularized_model.parameters(), lr=0.001, weight_decay=0.001)
print(regularized_model)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=8, out_features=1, bias=True)
)


## 13. Transfer learning
Transfer learning starts with a model trained on another dataset. Feature extraction freezes its learned backbone and trains a new output head; fine-tuning unfreezes some pretrained layers and adapts them with a small learning rate. It is useful when labeled data is limited and the source and target tasks are related.

In [14]:
feature_layer = nn.Linear(4, 8)
feature_layer.requires_grad_(False)
classifier = nn.Linear(8, 2)
transfer_model = nn.Sequential(feature_layer, nn.ReLU(), classifier)
print(transfer_model)
print('Trainable parameters:', sum(parameter.numel() for parameter in transfer_model.parameters() if parameter.requires_grad))

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=2, bias=True)
)
Trainable parameters: 18


## 14. Model evaluation
Accuracy is the fraction of predictions that are correct. Precision measures the reliability of positive predictions; recall measures the share of actual positives found; F1 balances precision and recall. A confusion matrix and per-class metrics expose errors hidden by overall accuracy. Compare training and validation metrics to diagnose generalization, and reserve the test set for final evaluation.

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
actual = [1, 1, 0, 0]
predicted = [1, 0, 1, 0]
print('Accuracy:', accuracy_score(actual, predicted))
print('Precision:', precision_score(actual, predicted))
print('Recall:', recall_score(actual, predicted))
print('F1:', f1_score(actual, predicted))
print('Confusion matrix:\n', confusion_matrix(actual, predicted))

Accuracy: 0.5
Precision: 0.5
Recall: 0.5
F1: 0.5
Confusion matrix:
 [[1 1]
 [1 1]]


## 15. Short debugging answers
If gradients grow across batches, clear them with `optimizer.zero_grad()` before backpropagation. `CrossEntropyLoss` expects raw logits, not softmax probabilities. `BCEWithLogitsLoss` expects float targets with the same shape as raw logits. For a matrix multiplication shape error, compare the input's final dimension with the layer's `in_features`. A suspiciously strong validation score can indicate data leakage; fit preprocessing only on training data and keep test data out of model selection.

## 16. Comparison answers
MAE is less sensitive to outliers than MSE; MSE emphasizes large errors. Batch gradient descent is stable but memory intensive, stochastic updates are noisy, and mini-batches balance both. Momentum accumulates gradient direction; Adam adapts parameter-wise step sizes. An ANN is a general choice for tabular features; a CNN is usually better for images because it preserves spatial structure with shared filters.